# Evaluation: Simulation-Pretrained Transformer vs RF Baseline

## Metrics
- ROC-AUC with bootstrap 95% CI (200 resamples, using split.bootstrap_roc_auc)
- PR-AUC (Average Precision)
- Best F1 threshold (selected on validation set)
- Confusion matrix at optimal threshold

## Comparisons
1. Simulation-pretrained + adversarial fine-tuned Transformer
2. RF baseline (16 physical features, AUC = 0.7994)
3. Ablation: Transformer trained from scratch on real data only (no pretraining)
4. Ablation: Simulation-pretrained WITHOUT domain adaptation (no adversarial)

## Fair comparison policy
All models select best checkpoint by val AUC (including ablations).
All models report PR-AUC and best-F1 threshold from validation set.

In [8]:
import os
os.environ["PYTHONHASHSEED"] = "0"

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, random
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR

seed = 42
random.seed(seed); np.random.seed(seed)
torch.manual_seed(seed); torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformer_model import ExoplanetTransformer, ExoplanetTransformerWithDomain

observations = pd.read_pickle("/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl")
from split import ensure_split, bootstrap_roc_auc
train_stars, val_stars, test_stars = ensure_split(observations)

NUM_FREQS = 8
PERIODS = np.logspace(np.log10(1.0), np.log10(7300.0), NUM_FREQS)
FREQS = 2.0 * np.pi / PERIODS

pos_stars = sorted(set(observations[observations["has_exoplanets"] == 1]["star_name"]))
neg_stars = sorted(set(observations[observations["has_exoplanets"] == 0]["star_name"]))

def star_to_features_real(star_name):
    star_obs = observations[observations["star_name"] == star_name].sort_values("bjd")
    ref_bjd = star_obs["bjd"].iloc[0]
    features = []
    for _, row in star_obs.iterrows():
        dt = row["bjd"] - ref_bjd
        pos_enc = []
        for f in FREQS: pos_enc.append(np.sin(f*dt)); pos_enc.append(np.cos(f*dt))
        features.append([row["rv_centered"], row["rv_err"], row["exposure_time"],
                        row["RHKp"], row["Halpha"]] + pos_enc)
    return np.array(features, dtype=np.float32)

test_pos = [s for s in test_stars if s in set(pos_stars)]
test_neg = [s for s in test_stars if s in set(neg_stars)]
val_pos = [s for s in val_stars if s in set(pos_stars)]
val_neg = [s for s in val_stars if s in set(neg_stars)]
train_pos = [s for s in train_stars if s in set(pos_stars)]
train_neg = [s for s in train_stars if s in set(neg_stars)]

sim_mean = np.load("/kaggle/working/sim_norm_mean.npy")
sim_std = np.load("/kaggle/working/sim_norm_std.npy")

train_data = [(star_to_features_real(s), 1) for s in train_pos] + [(star_to_features_real(s), 0) for s in train_neg]
train_data = [((seq - sim_mean) / sim_std, l) for seq, l in train_data]
val_data = [(star_to_features_real(s), 1) for s in val_pos] + [(star_to_features_real(s), 0) for s in val_neg]
val_data = [((seq - sim_mean) / sim_std, l) for seq, l in val_data]
test_data = [(star_to_features_real(s), 1) for s in test_pos] + [(star_to_features_real(s), 0) for s in test_neg]
test_data = [((seq - sim_mean) / sim_std, l) for seq, l in test_data]

from sim_dataset import collate_stars

class StarDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx][0], dtype=torch.float32), torch.tensor(self.data[idx][1], dtype=torch.float32)

train_loader = DataLoader(StarDataset(train_data), batch_size=32, shuffle=True, collate_fn=collate_stars, pin_memory=True)
val_loader = DataLoader(StarDataset(val_data), batch_size=32, shuffle=False, collate_fn=collate_stars)
test_loader = DataLoader(StarDataset(test_data), batch_size=32, shuffle=False, collate_fn=collate_stars)

n_pos = sum(l for _, l in train_data)
n_neg = len(train_data) - n_pos
pos_weight = torch.tensor([n_neg / max(n_pos, 1)]).to(device)

def train_with_val_checkpoint(model, tr_loader, v_loader, criterion, optimizer, n_epochs, label, lr_scheduler=None):
    best_val_auc = 0; best_state = None
    for epoch in range(n_epochs):
        model.train()
        for padded, mask, labels in tr_loader:
            padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(padded, mask)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        if lr_scheduler: lr_scheduler.step()
        model.eval()
        vp, vl = [], []
        with torch.no_grad():
            for padded, mask, labels in v_loader:
                padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
                logits = model(padded, mask)
                vp.extend(torch.sigmoid(logits).cpu().numpy())
                vl.extend(labels.cpu().numpy())
        val_auc = roc_auc_score(vl, vp)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f"  [{label}] Epoch {epoch+1}/{n_epochs} | Val AUC: {val_auc:.4f}")
    model.load_state_dict(best_state)
    print(f"  [{label}] Best val AUC: {best_val_auc:.4f}")
    return model

def evaluate_on_test(model, t_loader, v_loader):
    model.eval()
    vp, vl = [], []
    with torch.no_grad():
        for padded, mask, labels in v_loader:
            padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
            logits = model(padded, mask)
            vp.extend(torch.sigmoid(logits).cpu().numpy())
            vl.extend(labels.cpu().numpy())
    tp, tl = [], []
    with torch.no_grad():
        for padded, mask, labels in t_loader:
            padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
            logits = model(padded, mask)
            tp.extend(torch.sigmoid(logits).cpu().numpy())
            tl.extend(labels.cpu().numpy())
    auc, lo, hi = bootstrap_roc_auc(tl, tp, n_resamples=200, seed=42)
    pr = average_precision_score(tl, tp)
    prec_v, rec_v, thr_v = precision_recall_curve(vl, vp)
    f1_v = 2 * prec_v * rec_v / (prec_v + rec_v + 1e-8)
    best_idx = np.argmax(f1_v)
    best_thresh = thr_v[best_idx] if best_idx < len(thr_v) else 0.5
    preds = (np.array(tp) >= best_thresh).astype(int)
    cm = confusion_matrix(tl, preds)
    pt = cm[1,1] / (cm[1,1] + cm[0,1]) if cm[1,1] + cm[0,1] > 0 else 0
    rt = cm[1,1] / (cm[1,1] + cm[1,0]) if cm[1,1] + cm[1,0] > 0 else 0
    ft = 2 * pt * rt / (pt + rt) if pt + rt > 0 else 0
    return {"auc": auc, "ci_lo": lo, "ci_hi": hi, "pr_auc": pr, "f1": ft, "prec": pt, "rec": rt, "thresh": best_thresh, "cm": cm}

In [9]:
# Model 1: Simulation-pretrained + adversarial fine-tuned
print("Evaluating: Sim-Pretrained + Adversarial")
model_adv = ExoplanetTransformerWithDomain().to(device)
model_adv.load_state_dict(torch.load("/kaggle/working/finetuned_adversarial.pth"))
r1 = evaluate_on_test(model_adv, test_loader, val_loader)
print(f"  ROC-AUC: {r1['auc']:.4f} [{r1['ci_lo']:.4f}, {r1['ci_hi']:.4f}]")
print(f"  PR-AUC:  {r1['pr_auc']:.4f}")
print(f"  F1:      {r1['f1']:.4f} (P={r1['prec']:.4f}, R={r1['rec']:.4f}, thresh={r1['thresh']:.4f})")


Evaluating: Sim-Pretrained + Adversarial
  ROC-AUC: 0.5889 [0.5356, 0.6376]
  PR-AUC:  0.2062
  F1:      0.3212 (P=0.2304, R=0.5301, thresh=0.5344)


In [10]:
# Ablation 1: From scratch (no pretraining)
print("Training: From-scratch (50 epochs, val checkpoint)")
model_scratch = ExoplanetTransformer().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = Adam(model_scratch.parameters(), lr=1e-3, weight_decay=5e-3)
warmup, total = 5, 50
sched = LambdaLR(optimizer, lambda e: (e+1)/warmup if e < warmup else 0.5*(1+math.cos(math.pi*(e-warmup)/(total-warmup))))
model_scratch = train_with_val_checkpoint(model_scratch, train_loader, val_loader, criterion, optimizer, total, "scratch", sched)
r2 = evaluate_on_test(model_scratch, test_loader, val_loader)
print(f"  ROC-AUC: {r2['auc']:.4f} [{r2['ci_lo']:.4f}, {r2['ci_hi']:.4f}]")
print(f"  PR-AUC:  {r2['pr_auc']:.4f}")
print(f"  F1:      {r2['f1']:.4f} (P={r2['prec']:.4f}, R={r2['rec']:.4f}, thresh={r2['thresh']:.4f})")


Training: From-scratch (50 epochs, val checkpoint)
  [scratch] Epoch 1/50 | Val AUC: 0.5929
  [scratch] Epoch 10/50 | Val AUC: 0.7112
  [scratch] Epoch 20/50 | Val AUC: 0.7007
  [scratch] Epoch 30/50 | Val AUC: 0.7086
  [scratch] Epoch 40/50 | Val AUC: 0.7184
  [scratch] Epoch 50/50 | Val AUC: 0.7211
  [scratch] Best val AUC: 0.7317
  ROC-AUC: 0.6627 [0.5880, 0.7226]
  PR-AUC:  0.3411
  F1:      0.3786 (P=0.3171, R=0.4699, thresh=0.6804)


In [11]:
# Ablation 2: Sim-pretrained WITHOUT domain adaptation
print("Training: Sim-pretrained no-adv (50 epochs, val checkpoint)")
model_noadv = ExoplanetTransformerWithDomain().to(device)
pretrained = torch.load("/kaggle/working/pretrained_stage2.pth")
pk = {k: v for k, v in pretrained.items() if not k.startswith('classifier') and not k.startswith('domain_disc')}
model_noadv.load_state_dict(pk, strict=False)
criterion2 = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt2 = Adam(list(model_noadv.input_proj.parameters()) + list(model_noadv.transformer.parameters()) + list(model_noadv.pool.parameters()) + list(model_noadv.classifier.parameters()), lr=1e-4, weight_decay=5e-3)
sched2 = LambdaLR(opt2, lambda e: (e+1)/warmup if e < warmup else 0.5*(1+math.cos(math.pi*(e-warmup)/(total-warmup))))
model_noadv = train_with_val_checkpoint(model_noadv, train_loader, val_loader, criterion2, opt2, total, "no-adv", sched2)
r3 = evaluate_on_test(model_noadv, test_loader, val_loader)
print(f"  ROC-AUC: {r3['auc']:.4f} [{r3['ci_lo']:.4f}, {r3['ci_hi']:.4f}]")
print(f"  PR-AUC:  {r3['pr_auc']:.4f}")
print(f"  F1:      {r3['f1']:.4f} (P={r3['prec']:.4f}, R={r3['rec']:.4f}, thresh={r3['thresh']:.4f})")


Training: Sim-pretrained no-adv (50 epochs, val checkpoint)
  [no-adv] Epoch 1/50 | Val AUC: 0.5433
  [no-adv] Epoch 10/50 | Val AUC: 0.6774
  [no-adv] Epoch 20/50 | Val AUC: 0.7034
  [no-adv] Epoch 30/50 | Val AUC: 0.7069
  [no-adv] Epoch 40/50 | Val AUC: 0.7109
  [no-adv] Epoch 50/50 | Val AUC: 0.7106
  [no-adv] Best val AUC: 0.7116
  ROC-AUC: 0.6523 [0.5926, 0.7097]
  PR-AUC:  0.2730
  F1:      0.3825 (P=0.2857, R=0.5783, thresh=0.5037)


In [12]:
# Final comparison table
comparison = pd.DataFrame([
    {'Model': 'RF (16 physical)', 'ROC-AUC': 0.7994, '95% CI': '[TBD]', 'PR-AUC': 0.4658, 'Best F1': 0.5221, 'Input': 'aggregates'},
    {'Model': 'Transformer (scratch)', 'ROC-AUC': r2['auc'], '95% CI': f"[{r2['ci_lo']:.4f}, {r2['ci_hi']:.4f}]", 'PR-AUC': r2['pr_auc'], 'Best F1': r2['f1'], 'Input': 'sequences'},
    {'Model': 'Sim-Pretrained (no adapt)', 'ROC-AUC': r3['auc'], '95% CI': f"[{r3['ci_lo']:.4f}, {r3['ci_hi']:.4f}]", 'PR-AUC': r3['pr_auc'], 'Best F1': r3['f1'], 'Input': 'sequences'},
    {'Model': 'Sim-Pretrained+Adversarial', 'ROC-AUC': r1['auc'], '95% CI': f"[{r1['ci_lo']:.4f}, {r1['ci_hi']:.4f}]", 'PR-AUC': r1['pr_auc'], 'Best F1': r1['f1'], 'Input': 'sequences'},
])
print(comparison.to_string(index=False, float_format=lambda x: f'{x:.4f}' if not pd.isna(x) else '--'))

                     Model  ROC-AUC           95% CI  PR-AUC  Best F1      Input
          RF (16 physical)   0.7994            [TBD]  0.4658   0.5221 aggregates
     Transformer (scratch)   0.6627 [0.5880, 0.7226]  0.3411   0.3786  sequences
 Sim-Pretrained (no adapt)   0.6523 [0.5926, 0.7097]  0.2730   0.3825  sequences
Sim-Pretrained+Adversarial   0.5889 [0.5356, 0.6376]  0.2062   0.3212  sequences
